# Multi-Tool Agent with Text2Cypher

This notebook creates a comprehensive agent with three tools:
1. **Schema tool** - Database structure information
2. **Document search** - Semantic search with graph context
3. **Database query** - Text2Cypher for specific facts

The agent selects the appropriate tool based on the question type.

---

## Setup

Import required modules and configure connections.

In [ ]:
import sys
sys.path.insert(0, '..')

from strands import Agent, tool
from strands.models import BedrockModel
from neo4j_graphrag.retrievers import VectorCypherRetriever, Text2CypherRetriever
from neo4j_graphrag.schema import get_schema

from config import get_neo4j_driver, get_embedder, get_llm, aws_config

In [ ]:
driver = get_neo4j_driver()
driver.verify_connectivity()

embedder = get_embedder()
cypher_llm = get_llm()

print(f"Connected to Neo4j!")
print(f"Embedder: {embedder.model_id}")
print(f"Cypher LLM: {cypher_llm.model_id}")

## Create Retrievers

Set up both the VectorCypher and Text2Cypher retrievers.

In [ ]:
# Vector retriever with graph context
retrieval_query = """
MATCH (node)-[:FROM_DOCUMENT]-(doc:Document)-[:FILED]-(company:Company)
OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
WITH node, score, company, collect(risk.name)[0..10] AS risks
WHERE score IS NOT NULL
RETURN 
    node.text AS text,
    score,
    {company: company.name, risks: risks} AS metadata
ORDER BY score DESC
"""

vector_retriever = VectorCypherRetriever(
    driver=driver,
    index_name="chunkEmbeddings",
    embedder=embedder,
    retrieval_query=retrieval_query,
)

print("Vector retriever created!")

In [ ]:
# Custom prompt for modern Cypher generation
cypher_prompt = """Task: Generate a Cypher statement to query a graph database.

Instructions:
- Use only the provided relationship types and properties in the schema.
- Do not use any other relationship types or properties that are not provided.
- Only filter by name when a specific entity name is mentioned in the question.
  When filtering by name, use case-insensitive matching:
  `WHERE toLower(node.name) CONTAINS toLower('ActualEntityName')`
- Do NOT add name filters if no specific entity name is mentioned in the question.
- Always add LIMIT 20 to the end of your query to restrict results.

Modern Cypher Requirements:
- Use `elementId(node)` instead of `id(node)` (id() is removed in Neo4j 5+).
- Use `count{{pattern}}` instead of `size((pattern))` for counting patterns.
- Use `EXISTS {{MATCH pattern}}` instead of `exists((pattern))` for existence checks.
- When using ORDER BY, filter NULL values first: `WHERE property IS NOT NULL ORDER BY property`.
- Use explicit grouping with WITH clauses for aggregations.
- Limit collected results when appropriate: `collect(item)[0..20]`.

Schema:
{schema}

Note: Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.

The question is:
{query_text}"""

text2cypher_retriever = Text2CypherRetriever(
    driver=driver,
    llm=cypher_llm,
    neo4j_schema=get_schema(driver),
    custom_prompt=cypher_prompt,
)

print("Text2Cypher retriever created!")

## Define Agent Tools

Create three specialized tools for the agent.

In [ ]:
@tool
def get_graph_schema() -> str:
    """
    Get the schema of the graph database including node labels,
    relationship types, and properties.
    
    Use this tool when the user asks about:
    - What data is in the database
    - The structure of the graph
    - What types of nodes or relationships exist
    
    Returns:
        A string describing the database schema.
    """
    return get_schema(driver)


@tool
def retrieve_financial_documents(query: str) -> str:
    """
    Find details about companies in their financial documents using semantic search.
    
    This tool searches SEC 10-K filing documents and returns relevant passages
    along with the associated company name and risk factors.
    
    Use this tool when the user asks about:
    - General company information or descriptions
    - Products, services, or business strategies
    - Narrative content from filings
    - Qualitative information
    
    Args:
        query: The search query to find relevant documents.
        
    Returns:
        Relevant document passages with company and risk context.
    """
    try:
        results = vector_retriever.search(query_text=query, top_k=3)
        if not results.items:
            return "No documents found matching the query."
        return "\n\n".join(str(item.content) for item in results.items)
    except Exception as e:
        return f"Error searching documents: {e}"


@tool
def query_database(question: str) -> str:
    """
    Get specific facts from the database by generating and executing a Cypher query.
    
    This tool converts natural language questions into database queries to find
    specific entities, counts, or relationships.
    
    Use this tool when the user asks for:
    - Specific facts: "What companies are owned by BlackRock?"
    - Counts or aggregations: "How many risk factors does Apple face?"
    - Lists of entities: "What asset managers are in the database?"
    - Relationship queries: "Which companies share risks?"
    
    Args:
        question: A natural language question about the data.
        
    Returns:
        Database query results as a string.
    """
    try:
        results = text2cypher_retriever.search(query_text=question)
        if not results.items:
            return "No results found for the query."
        return "\n\n".join(str(item.content) for item in results.items)
    except Exception as e:
        return f"Error querying database: {e}"


print("Tools defined: get_graph_schema, retrieve_financial_documents, query_database")

## Create the Multi-Tool Agent

Create an agent with all three tools and clear instructions.

In [ ]:
bedrock_model = BedrockModel(
    model_id=aws_config.bedrock_model_id,
    region_name=aws_config.region,
    temperature=0.3,
)

agent = Agent(
    model=bedrock_model,
    system_prompt="""
    You are a helpful assistant that answers questions about a Neo4j graph database
    containing SEC 10-K financial filings from major companies.
    
    You have three tools:
    
    1. get_graph_schema - Use when asked about database structure or what data exists
    
    2. retrieve_financial_documents - Use for semantic search of document content.
       Good for questions about products, services, business descriptions,
       and narrative content.
    
    3. query_database - Use for specific facts, counts, and entity lookups.
       Good for questions like "What companies does X own?" or 
       "How many risk factors does Y have?"
    
    Choose the most appropriate tool based on the question type.
    When a tool returns data, use that data to answer the question directly.
    Be concise and accurate.
    """,
    tools=[get_graph_schema, retrieve_financial_documents, query_database]
)

print("Multi-tool agent created!")

## Test Text2Cypher Queries

These questions should use the `query_database` tool for specific facts.

In [ ]:
query = "What companies are owned by BlackRock Inc?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

In [ ]:
query = "Who are the asset managers in the database?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

In [ ]:
query = "Which company faces the most risk factors?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

## Test Semantic Search Queries

These questions should use the `retrieve_financial_documents` tool.

In [ ]:
query = "What does NVIDIA say about AI and machine learning in their filings?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

In [ ]:
query = "Summarize Apple's business strategy from their financial documents."

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

## Test Schema Queries

These questions should use the `get_graph_schema` tool.

In [ ]:
query = "How does the graph model relate financial documents to companies and risks?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

## Summary

In this notebook, you built a comprehensive GraphRAG agent with:

1. **Three specialized tools** for different question types
2. **Intelligent tool selection** based on the query
3. **Text2Cypher** for specific facts and counts
4. **Semantic search** for document content
5. **Schema exploration** for structural questions

### Tool Selection Summary

| Question Type | Tool Used | Examples |
|--------------|-----------|----------|
| Structure | get_graph_schema | "What data is in the database?" |
| Content | retrieve_financial_documents | "What products does Apple make?" |
| Facts | query_database | "Who owns Microsoft?" |

### Key Takeaways

- **Clear docstrings** are critical for tool selection
- **Multiple tools** allow the agent to handle diverse questions
- **Strands SDK** makes it easy to build capable agents
- **Amazon Bedrock + Claude** provides powerful reasoning

---

**Congratulations!** You've completed the GraphRAG Agents lab. Continue to [Lab 9 - Hybrid Search](../Lab_9_Hybrid_Search/) for advanced search patterns.

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")